# Scrape Hindi and Nepali news on Kaggle

Collects the **manual** portion of both corpora - the requirement neither
language can currently meet.

## Why this belongs on Kaggle

Crawling is rate-limited to 1 request per second per host, so it is pure
waiting rather than compute. Moving it here frees the local machine for the
corpus builds, which are genuinely CPU-bound.

## Before running

**Settings → Internet → On.** Off by default; nothing works without it.

No GitHub token needed - the crawler code is embedded below.

## Sessions

Kaggle caps a session at 12 hours and the full target needs ~36 hours of
crawling, so expect 3 sessions - or run several notebooks in parallel with
`SITES_FILTER` set to different sites in each.

Nothing is lost between sessions: the crawler appends to its output and caches
every page it fetches, so a re-run only pays for URLs it has not seen.
**Save Version → Save & Run All (Commit)** to run unattended.


## 1 · Setup


In [ ]:
import os, sys, pathlib, subprocess, shutil

REPO = pathlib.Path('/kaggle/working')
sys.path.insert(0, str(REPO))

probe = subprocess.run(['curl', '-sS', '-m', '10', '-o', '/dev/null',
                        '-w', '%{http_code}', 'https://www.google.com'],
                       capture_output=True, text=True)
assert probe.stdout.strip() == '200', 'Enable Internet in Settings, then re-run.'
print('internet OK')
print(f'disk free: {shutil.disk_usage(REPO).free / 1e9:.1f} GB')


In [ ]:
%pip install -q beautifulsoup4 lxml pyyaml 2>&1 | tail -1
print('deps installed')


## 2 · Write the crawler

Only the modules scraping needs: the source definitions, the script-aware
normalizer, and the language identifier that verifies each page is really in
the language claimed.


In [ ]:
import base64, gzip, json, pathlib

_BLOB = (
    "..."
)

_files = json.loads(gzip.decompress(base64.b64decode(_BLOB)).decode('utf-8'))
for _rel, _content in _files.items():
    _p = REPO / _rel
    _p.parent.mkdir(parents=True, exist_ok=True)
    _p.write_text(_content, encoding='utf-8')
print(f'wrote {len(_files)} files')


## 3 · Choose what to crawl

`LANGUAGE` picks the manifest. `SITES_FILTER` narrows it to particular sites -
use it to split the work across parallel notebooks, e.g. `['bhaskar', 'jagran']`
in one and `['aajtak', 'abplive']` in another.

Leave `SITES_FILTER` empty to crawl every site in the manifest.


In [ ]:
LANGUAGE = 'hi'          # 'hi' for Hindi, 'ne' for Nepali
SITES_FILTER = []        # e.g. ['bhaskar', 'jagran']; empty = all sites
MAX_PAGES_PER_SITE = 30000

import yaml

lang_dir = 'hindi' if LANGUAGE == 'hi' else 'nepali'
manifest = REPO / f'{lang_dir}/configs/sources.yaml'
cfg = yaml.safe_load(manifest.read_text())

kept = []
for src in cfg['sources']:
    if src.get('type') != 'scrape' or not src.get('enabled', True):
        continue
    if SITES_FILTER and not any(f in src['source_id'] for f in SITES_FILTER):
        continue
    src['max_pages'] = MAX_PAGES_PER_SITE
    src['cache_dir'] = str(REPO / f'{lang_dir}/scrape_cache')
    kept.append(src)

cfg['sources'] = kept
manifest.write_text(yaml.safe_dump(cfg, allow_unicode=True, sort_keys=False))

print(f'{len(kept)} site(s) to crawl:')
for s in kept:
    print(f"  {s['source_id']:32s} {s['start_urls'][0]}")


## 4 · Crawl

**The long cell.** Runs until every site hits `MAX_PAGES_PER_SITE` or the
session ends. Progress prints every 100 pages.

Expect roughly **750 words per Hindi article** and **215 per Nepali** - the gap
is why Hindi reaches its target far sooner.


In [ ]:
OUTPUT = REPO / f'{lang_dir}_news.jsonl'

cmd = [sys.executable, str(REPO / 'scripts/scrape_only.py'),
       '--config', str(manifest), '--output', str(OUTPUT), '--report-every', '100']

proc = subprocess.Popen(cmd, cwd=str(REPO), stdout=subprocess.PIPE,
                        stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    print(line, end='')
print('exit code:', proc.wait())


## 5 · Check what was collected

Verifies the pages really are in the target language - worth doing for Hindi
and Nepali especially, since they share the Devanagari script and a naive
crawl of one can pick up the other.


In [ ]:
import json
from lma.data.language_id import IndicLanguageIdentifier
from lma.data.normalize import IndicNormalizer, get_profile

ident = IndicLanguageIdentifier()
norm = IndicNormalizer(get_profile(LANGUAGE))

pages = words = 0
verdicts = {}
with open(OUTPUT, encoding='utf-8') as fh:
    for i, line in enumerate(fh):
        try:
            text = json.loads(line)['text']
        except Exception:
            continue
        pages += 1
        words += len(text.split())
        if i < 3000:
            v = ident.identify(norm.normalize(text)).language
            verdicts[v] = verdicts.get(v, 0) + 1

TARGET_WORDS = 52_800_000   # 20% of 500M tokens, at ~1.9 tokens/word
print(f'pages : {pages:,}')
print(f'words : {words:,}  ({100 * words / TARGET_WORDS:.1f}% of the manual target)')
print(f'language verdicts (first 3k): {verdicts}')


## 6 · Download the result

**Save Version → Save & Run All (Commit)**, then take the JSONL from the
notebook's **Output** tab.

Locally, drop it into `<language>/data/raw/scraped/` and the next corpus build
picks it up automatically as manual collection - the `collection: manual` label
travels inside the file, so nothing else needs configuring.


In [ ]:
print(f'{OUTPUT}  ({OUTPUT.stat().st_size / 1e6:.1f} MB)')
print()
print('Next session: re-run this notebook. The crawler appends and the page')
print('cache persists in /kaggle/working, so it resumes rather than restarts.')
